# LeetCode 4086 — Minimum Interval Selection to Cover Target Intervals (Hard)

Bài toán LeetCode:

```python
class Solution:
    def minSelection(self, availableIntervals: List[List[int]], targetIntervals: List[List[int]]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4086 (Weekly Contest 524 — câu 4, rating khoảng 2202).

Đây là một biến thể của bài toán kinh điển **"phủ đoạn bằng ít đoạn nhất" (interval covering)** — họ hàng của *Video Stitching* và *Jump Game II*. Điểm mới:

- cần phủ **nhiều** đoạn mục tiêu, có thể chồng nhau hoặc cách xa nhau;
- phải phủ **mọi số thực** trong mục tiêu — nên `[1, 2]` và `[3, 4]` **không** nối liền (điểm `2.5` bị hở).

Kỹ thuật chính: **sắp xếp + tham lam "đi xa nhất có thể"**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đoạn đóng, phủ số thực, và vì sao "chạm nhau" khác "cách nhau 1 đơn vị".
2. Gộp các đoạn chồng nhau (merge intervals).
3. Brute Force thử mọi tập con.
4. Chiến lược tham lam: luôn chọn đoạn **vươn xa nhất** trong các đoạn chứa điểm chưa phủ — và vì sao nó tối ưu.
5. Cài đặt một con trỏ `O(n log n + m log m)` và cách dùng mảng "tầm với" theo tọa độ.
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho hai danh sách **đoạn đóng** (gồm cả hai đầu mút):

- `availableIntervals`: các đoạn có sẵn để chọn;
- `targetIntervals`: các đoạn mục tiêu cần phủ.

Chọn **ít đoạn có sẵn nhất** sao cho **mọi số thực** nằm trong **bất kỳ** đoạn mục tiêu nào đều thuộc ít nhất một đoạn đã chọn. Điểm nằm ngoài các mục tiêu không cần phủ. Các đoạn được chọn có thể chồng nhau hoặc thò ra ngoài mục tiêu.

Trả về số đoạn ít nhất, hoặc `-1` nếu không thể phủ.

---

## Ví dụ 1

```python
available = [[1,4],[1,7],[3,9],[5,10],[10,15]]
target    = [[1,3],[2,5],[7,8],[10,13],[11,15],[12,13]]
-> 3
```

Các mục tiêu gộp lại thành `[1, 5]`, `[7, 8]`, `[10, 15]`. Chọn `[1, 4]`, `[3, 9]`, `[10, 15]` là phủ hết (còn nhiều cách khác cũng dùng 3 đoạn).

## Ví dụ 2

```python
available = [[4,6],[1,4],[2,5]], target = [[3,4],[2,5],[6,7]]   ->   -1
```

Các điểm ngay sau `6` trong `[6, 7]` (ví dụ `6.5`) không đoạn nào phủ.

## Ví dụ 3

```python
available = [[1,2],[3,4]], target = [[1,4]]   ->   -1
```

Điểm `2.5` nằm trong mục tiêu nhưng không thuộc đoạn có sẵn nào.

---

## Ràng buộc

```text
1 <= availableIntervals.length, targetIntervals.length <= 5 * 10^4
1 <= left <= right <= 10^5   (cho cả hai danh sách)
```

# 2. Kiến thức nền tảng

## 2.1. Đoạn đóng và phủ số thực

`[a, b]` gồm mọi số thực `x` với `a <= x <= b`. Hai đoạn đóng `[a, b]` và `[c, d]` (với `a <= c`) có hợp **liền một khối** khi và chỉ khi `c <= b` — tức là **chạm hoặc chồng** nhau:

- `[1, 2]` và `[2, 3]` → hợp là `[1, 3]` ✔ (điểm 2 thuộc cả hai)
- `[1, 2]` và `[3, 4]` → hở khoảng `(2, 3)` ✗

Cẩn thận: nếu nghĩ theo **số nguyên** thì `[1, 2]` và `[3, 4]` "phủ hết 1, 2, 3, 4" — nhưng đề nói về **số thực**.

## 2.2. Gộp đoạn (merge intervals)

Sắp theo đầu trái, duyệt và gộp đoạn mới vào đoạn cuối nếu `đầu trái mới <= đầu phải cũ`.

In [1]:
from typing import List


def merge_intervals(intervals: List[List[int]]) -> List[List[int]]:
    merged = []
    for a, b in sorted(intervals):
        if merged and a <= merged[-1][1]:          # chạm hoặc chồng -> gộp
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    return merged


print(merge_intervals([[1, 3], [2, 5], [7, 8], [10, 13], [11, 15], [12, 13]]))
print(merge_intervals([[1, 2], [2, 3]]))     # chạm nhau -> [[1, 3]]
print(merge_intervals([[1, 2], [3, 4]]))     # cách nhau -> giữ riêng

[[1, 5], [7, 8], [10, 15]]
[[1, 3]]
[[1, 2], [3, 4]]


## 2.3. Kiểm tra một tập đoạn đã chọn có phủ hết mục tiêu không

Gộp các đoạn đã chọn; mỗi mục tiêu `[l, r]` phải nằm **trọn trong một** khối đã gộp.

In [2]:
def covers(chosen, targets) -> bool:
    blocks = merge_intervals([list(iv) for iv in chosen])
    return all(any(a <= l and r <= b for a, b in blocks) for l, r in targets)


print(covers([[1, 4], [3, 9], [10, 15]], [[1, 3], [2, 5], [7, 8], [10, 13], [11, 15], [12, 13]]))
print(covers([[1, 2], [3, 4]], [[1, 4]]))

True
False


# 3. Cách 1 — Brute Force: thử mọi tập con theo kích thước tăng dần

In [3]:
from itertools import combinations


def min_selection_bruteforce(available: List[List[int]], targets: List[List[int]]) -> int:
    for k in range(len(available) + 1):
        for chosen in combinations(available, k):
            if covers(chosen, targets):
                return k
    return -1


ex1 = ([[1, 4], [1, 7], [3, 9], [5, 10], [10, 15]], [[1, 3], [2, 5], [7, 8], [10, 13], [11, 15], [12, 13]])
ex2 = ([[4, 6], [1, 4], [2, 5]], [[3, 4], [2, 5], [6, 7]])
ex3 = ([[1, 2], [3, 4]], [[1, 4]])
for ex in (ex1, ex2, ex3):
    print(min_selection_bruteforce(*ex))

3
-1
-1


`2^n` tập con — chỉ dùng được khi `n` rất nhỏ.

---

# 4. Bước đột phá: tham lam "đi xa nhất có thể"

Gọi `cur` = **điểm xa nhất đã phủ liên tục** (trên khối mục tiêu đang xử lý). Xử lý các mục tiêu theo đầu trái tăng dần.

Với mục tiêu `[l, r]` mà `cur < r` (còn phần chưa phủ), điểm **chưa phủ đầu tiên** là:

```text
unc = max(l, cur)
```

(nếu `l > cur` thì có khoảng trống giữa các mục tiêu — bỏ qua được, điểm cần phủ tiếp theo là `l`; ngược lại phải phủ tiếp ngay từ `cur`).

Ta **bắt buộc** phải chọn một đoạn chứa điểm `unc`, tức đoạn có `left <= unc` (và `right >= unc`). Trong các đoạn đó, nên chọn đoạn nào?

> **Chọn đoạn có `right` lớn nhất.**

**Lập luận hoán đổi (exchange argument):** giả sử một lời giải tối ưu nào đó dùng đoạn `[a, b]` để phủ `unc`. Đoạn tham lam `[a', b']` cũng chứa `unc` và có `b' >= b`. Thay `[a, b]` bằng `[a', b']`: phần `[unc, b]` vẫn được phủ (vì `a' <= unc` và `b' >= b`), phần trước `unc` đã được phủ sẵn, nên lời giải vẫn hợp lệ với **cùng số đoạn**. Vậy tồn tại lời giải tối ưu bắt đầu bằng lựa chọn tham lam.

Sau khi chọn: `cur = b'`. Nếu đoạn xa nhất có `right < unc` (không đoạn nào chứa `unc`) hoặc `right == cur` (không tiến thêm được khi `unc = cur`) → **không thể phủ** → `-1`.

## Một con trỏ duy nhất

Sắp các đoạn có sẵn theo đầu trái. Vì `unc` **không bao giờ giảm**, tập "đoạn có `left <= unc`" chỉ lớn dần → một con trỏ `i` chạy tới, giữ `nxt` = `right` lớn nhất đã gặp (khởi tạo bằng `cur`).

# 5. Cách 2 — Gộp mục tiêu rồi tham lam từng khối (dễ hiểu)

Trước hết, cài đặt "nguyên văn" ý tưởng: gộp mục tiêu thành các khối rời nhau, rồi với mỗi khối `[L, R]` chạy tham lam, mỗi bước **quét lại toàn bộ** danh sách đoạn để tìm đoạn vươn xa nhất. Lưu ý quan trọng: biến `cur` phải **giữ nguyên qua các khối** — một đoạn đã chọn có thể phủ luôn cả khối sau (ví dụ đoạn `[1, 9]` phủ cả khối `[1, 3]` lẫn `[6, 7]`). Dễ viết, `O(m log m + đáp án · n)`.

In [4]:
def min_selection_simple(available: List[List[int]], targets: List[List[int]]) -> int:
    ans = 0
    cur = 0                                 # điểm xa nhất đã phủ (giữ qua các khối!)
    for L, R in merge_intervals(targets):
        while cur < R:                      # khối đã bị phủ sẵn thì bỏ qua
            unc = max(L, cur)               # điểm chưa phủ đầu tiên của khối
            best = -1
            for a, b in available:          # quét lại TOÀN BỘ danh sách
                if a <= unc:
                    best = max(best, b)
            if best < unc or best <= cur:
                return -1
            cur = best
            ans += 1
    return ans


for ex in (ex1, ex2, ex3):
    print(min_selection_simple(*ex))

3
-1
-1


# 6. Cách 3 — Sắp xếp + một con trỏ: O(n log n + m log m)

Không cần gộp mục tiêu trước: chỉ cần sắp mục tiêu theo đầu trái; mục tiêu nằm gọn trong phần đã phủ (`r <= cur`) thì bỏ qua.

In [5]:
def min_selection_greedy(available: List[List[int]], targets: List[List[int]]) -> int:
    avail = sorted(available)
    cur = 0                                  # điểm xa nhất đã phủ (0: chưa phủ gì, tọa độ >= 1)
    i = 0
    ans = 0
    for l, r in sorted(targets):
        while cur < r:
            unc = max(l, cur)                # điểm chưa phủ đầu tiên
            nxt = cur
            while i < len(avail) and avail[i][0] <= unc:
                nxt = max(nxt, avail[i][1])
                i += 1
            if nxt < unc or nxt == cur:      # không đoạn nào chứa unc / không tiến được
                return -1
            cur = nxt
            ans += 1
    return ans


for ex in (ex1, ex2, ex3):
    print(min_selection_greedy(*ex))

3
-1
-1


## Mô phỏng ví dụ 1

Đoạn có sẵn sau khi sắp: `[1,4], [1,7], [3,9], [5,10], [10,15]`. Mục tiêu sau khi sắp: `[1,3], [2,5], [7,8], [10,13], [11,15], [12,13]`.

| Mục tiêu | cur trước | unc | các đoạn được quét thêm | nxt | chọn | cur sau | ans |
|---|---:|---:|---|---:|:---:|---:|---:|
| `[1,3]` | 0 | 1 | `[1,4]`, `[1,7]` | 7 | ✔ | 7 | 1 |
| `[2,5]` | 7 | — | (đã phủ, bỏ qua) | | | 7 | 1 |
| `[7,8]` | 7 | 7 | `[3,9]`, `[5,10]` | 10 | ✔ | 10 | 2 |
| `[10,13]` | 10 | 10 | `[10,15]` | 15 | ✔ | 15 | 3 |
| `[11,15]`, `[12,13]` | 15 | — | (đã phủ) | | | 15 | 3 |

Đáp án `3`. Tham lam chọn `[1,7]`, `[5,10]`, `[10,15]` — khác với lựa chọn trong đề nhưng cùng số đoạn.

In [6]:
def trace_greedy(available: List[List[int]], targets: List[List[int]]) -> int:
    avail = sorted(available)
    cur, i, ans = 0, 0, 0
    for l, r in sorted(targets):
        if cur >= r:
            print(f"mục tiêu [{l},{r}]: đã phủ (cur = {cur})")
            continue
        while cur < r:
            unc = max(l, cur)
            nxt = cur
            scanned = []
            while i < len(avail) and avail[i][0] <= unc:
                scanned.append(avail[i])
                nxt = max(nxt, avail[i][1])
                i += 1
            if nxt < unc or nxt == cur:
                print(f"mục tiêu [{l},{r}]: unc = {unc}, không đoạn nào phủ tiếp -> -1")
                return -1
            ans += 1
            print(f"mục tiêu [{l},{r}]: unc = {unc}, quét thêm {scanned}, chọn đoạn tới {nxt} "
                  f"-> cur = {nxt}, ans = {ans}")
            cur = nxt
    return ans


trace_greedy(*ex1)
print()
trace_greedy(*ex2)

mục tiêu [1,3]: unc = 1, quét thêm [[1, 4], [1, 7]], chọn đoạn tới 7 -> cur = 7, ans = 1
mục tiêu [2,5]: đã phủ (cur = 7)
mục tiêu [7,8]: unc = 7, quét thêm [[3, 9], [5, 10]], chọn đoạn tới 10 -> cur = 10, ans = 2
mục tiêu [10,13]: unc = 10, quét thêm [[10, 15]], chọn đoạn tới 15 -> cur = 15, ans = 3
mục tiêu [11,15]: đã phủ (cur = 15)
mục tiêu [12,13]: đã phủ (cur = 15)

mục tiêu [2,5]: unc = 2, quét thêm [[1, 4], [2, 5]], chọn đoạn tới 5 -> cur = 5, ans = 1
mục tiêu [3,4]: đã phủ (cur = 5)
mục tiêu [6,7]: unc = 6, quét thêm [[4, 6]], chọn đoạn tới 6 -> cur = 6, ans = 2
mục tiêu [6,7]: unc = 6, không đoạn nào phủ tiếp -> -1


-1

Ở ví dụ 2: sau khi phủ tới `cur = 6`, mục tiêu `[6, 7]` cần phủ tiếp từ `unc = 6`; không còn đoạn nào có `left <= 6` mà `right > 6` → `-1`.

# 7. Cách 4 — Mảng "tầm với" theo tọa độ

Vì tọa độ `<= 10^5`, có thể thay việc sắp xếp đoạn có sẵn bằng một mảng:

```text
reach[x] = right lớn nhất của các đoạn có left <= x
```

Tính bằng cách ghi `reach[left] = max(reach[left], right)` rồi lấy **max tiền tố**. Khi đó "đoạn vươn xa nhất chứa `unc`" chính là `reach[unc]` — tra trong `O(1)`. Đây là ý tưởng của bài *Jump Game II*.

In [7]:
def min_selection_reach(available: List[List[int]], targets: List[List[int]]) -> int:
    M = max(max(b for _, b in available), max(b for _, b in targets)) + 1
    reach = [-1] * (M + 1)
    for a, b in available:
        reach[a] = max(reach[a], b)
    for x in range(1, M + 1):
        reach[x] = max(reach[x], reach[x - 1])        # max tiền tố
    cur, ans = 0, 0
    for l, r in sorted(targets):
        while cur < r:
            unc = max(l, cur)
            nxt = reach[unc]
            if nxt < unc or nxt <= cur:
                return -1
            cur = nxt
            ans += 1
    return ans


for ex in (ex1, ex2, ex3):
    print(min_selection_reach(*ex))

3
-1
-1


# 8. Phiên bản đúng format LeetCode

In [8]:
from typing import List


class Solution:
    def minSelection(self, availableIntervals: List[List[int]], targetIntervals: List[List[int]]) -> int:
        avail = sorted(availableIntervals)
        cur = 0
        i = 0
        ans = 0
        for l, r in sorted(targetIntervals):
            while cur < r:
                unc = max(l, cur)
                nxt = cur
                while i < len(avail) and avail[i][0] <= unc:
                    if avail[i][1] > nxt:
                        nxt = avail[i][1]
                    i += 1
                if nxt < unc or nxt == cur:
                    return -1
                cur = nxt
                ans += 1
        return ans

In [9]:
sol = Solution()
tests = [
    (ex1[0], ex1[1], 3, "ví dụ 1"),
    (ex2[0], ex2[1], -1, "ví dụ 2"),
    (ex3[0], ex3[1], -1, "ví dụ 3: hở (2, 3)"),
    ([[1, 2], [2, 4]], [[1, 4]], 2, "chạm nhau tại 2: liền"),
    ([[5, 5]], [[5, 5]], 1, "mục tiêu là một điểm"),
    ([[1, 3]], [[4, 4]], -1, "điểm không được phủ"),
    ([[1, 10]], [[2, 3], [5, 6], [8, 9]], 1, "một đoạn phủ cả ba"),
    ([[2, 5], [1, 3], [4, 8], [6, 9]], [[2, 4], [7, 9]], 2, "bài tập 1"),
]
for a, t, expected, name in tests:
    result = sol.minSelection(a, t)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:28} -> {result}")

PASS | ví dụ 1                      -> 3
PASS | ví dụ 2                      -> -1
PASS | ví dụ 3: hở (2, 3)           -> -1
PASS | chạm nhau tại 2: liền        -> 2
PASS | mục tiêu là một điểm         -> 1
PASS | điểm không được phủ          -> -1
PASS | một đoạn phủ cả ba           -> 1
PASS | bài tập 1                    -> 2


# 9. Vì sao tham lam đúng?

1. **Bất biến:** sau mỗi lựa chọn, mọi điểm mục tiêu `<= cur` đã được phủ, và tồn tại một lời giải tối ưu bắt đầu bằng đúng các đoạn tham lam đã chọn.
2. **Bắt buộc phủ `unc`:** điểm `unc` là điểm mục tiêu chưa phủ nhỏ nhất. Bất kỳ lời giải nào cũng phải có một đoạn chứa `unc`.
3. **Hoán đổi:** thay đoạn đó bằng đoạn chứa `unc` vươn xa nhất không làm hỏng lời giải và không tăng số đoạn (mục 4). Lặp lại lập luận cho từng bước → tham lam tối ưu.
4. **Phát hiện không thể:** nếu đoạn xa nhất có `right < unc` (không đoạn nào chứa `unc`), hoặc khi `unc = cur` mà `right == cur` (các điểm ngay sau `cur` không đoạn nào chứa), thì có điểm mục tiêu không thể phủ.
5. **Một con trỏ đủ:** `unc` không giảm, nên đoạn đã quét (có `left <= unc` cũ) vẫn có `left <= unc` mới; giá trị `right` lớn nhất trong số đó hoặc đã được chọn, hoặc `<= cur`.

# 10. Độ phức tạp

Gọi `n` = số đoạn có sẵn, `m` = số mục tiêu, `V` = tọa độ lớn nhất.

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | thử mọi tập con | `O(2^n · n log n)` | `O(n)` |
| 2. Gộp + quét lại | tham lam, mỗi bước quét hết | `O(m log m + ans · n)` | `O(m)` |
| 3. Một con trỏ | tham lam, con trỏ chỉ tiến | `O(n log n + m log m)` | `O(1)` thêm (ngoài sắp xếp) |
| 4. Mảng tầm với | `reach[x]` = max tiền tố | `O(n + m log m + V)` | `O(V)` |

# 11. Đối chiếu bốn cách trên dữ liệu ngẫu nhiên

Sinh dữ liệu nhỏ (tọa độ 1..9) để có nhiều trường hợp chạm/hở/điểm đơn.

In [10]:
import random

random.seed(4086)
count_impossible = 0
for _ in range(3000):
    av = []
    for _ in range(random.randint(1, 6)):
        a = random.randint(1, 8)
        av.append([a, random.randint(a, 9)])
    tg = []
    for _ in range(random.randint(1, 4)):
        a = random.randint(1, 8)
        tg.append([a, random.randint(a, 9)])
    expected = min_selection_bruteforce(av, tg)
    assert min_selection_simple(av, tg) == expected, (av, tg)
    assert min_selection_greedy(av, tg) == expected, (av, tg)
    assert min_selection_reach(av, tg) == expected, (av, tg)
    assert sol.minSelection(av, tg) == expected, (av, tg)
    count_impossible += expected == -1
print(f"3000 bộ test khớp brute force ({count_impossible} bộ không thể phủ)")

3000 bộ test khớp brute force (1835 bộ không thể phủ)


In [11]:
import time

N = 5 * 10**4
big_av = []
for _ in range(N):
    a = random.randint(1, 10**5 - 50)
    big_av.append([a, a + random.randint(0, 50)])
big_tg = []
for _ in range(N):
    a = random.randint(1, 10**5 - 5)
    big_tg.append([a, a + random.randint(0, 5)])
t0 = time.perf_counter()
print("kết quả:", sol.minSelection(big_av, big_tg), f"({time.perf_counter() - t0:.2f}s)")

kết quả: 2572 (0.06s)


# 12. Những lỗi dễ mắc

## Lỗi 1 — Nghĩ theo số nguyên: coi `[1, 2]` và `[3, 4]` là liền nhau

In [12]:
def wrong_integer_thinking(available, targets):
    avail = sorted(available)
    cur, i, ans = 0, 0, 0
    for l, r in sorted(targets):
        while cur < r:
            unc = max(l, cur + 1)                     # SAI: nhảy sang số nguyên kế tiếp
            nxt = cur
            while i < len(avail) and avail[i][0] <= unc:
                nxt = max(nxt, avail[i][1])
                i += 1
            if nxt < unc:
                return -1
            cur = nxt
            ans += 1
    return ans


print("Sai :", wrong_integer_thinking(*ex3))
print("Đúng:", sol.minSelection(*ex3))

Sai : 2
Đúng: -1


## Lỗi 2 — Thiếu điều kiện `nxt == cur` → vòng lặp vô hạn

Khi `unc = cur` và không có đoạn nào vươn quá `cur`, `nxt` vẫn bằng `cur`. Nếu chỉ kiểm tra `nxt < unc`, vòng `while cur < r` lặp mãi mãi. Cell dưới giới hạn số vòng để minh họa.

In [13]:
def wrong_no_progress_check(available, targets, max_iter=20):
    avail = sorted(available)
    cur, i, ans, it = 0, 0, 0, 0
    for l, r in sorted(targets):
        while cur < r:
            it += 1
            if it > max_iter:
                return f"lặp quá {max_iter} vòng (vô hạn!)"
            unc = max(l, cur)
            nxt = cur
            while i < len(avail) and avail[i][0] <= unc:
                nxt = max(nxt, avail[i][1])
                i += 1
            if nxt < unc:                              # SAI: thiếu "or nxt == cur"
                return -1
            cur = nxt
            ans += 1
    return ans


print("Sai :", wrong_no_progress_check(*ex2))
print("Đúng:", sol.minSelection(*ex2))

Sai : lặp quá 20 vòng (vô hạn!)
Đúng: -1


## Lỗi 3 — Chọn đoạn theo thứ tự xuất hiện / đoạn dài nhất

Chọn "đoạn dài nhất" hay "đoạn đầu tiên chứa `unc`" đều có thể dùng nhiều đoạn hơn. Tiêu chí đúng là **vươn xa nhất về bên phải** trong số các đoạn **chứa** điểm chưa phủ.

In [14]:
def wrong_first_fit(available, targets):
    cur, ans = 0, 0
    for l, r in sorted(targets):
        while cur < r:
            unc = max(l, cur)
            cand = [iv for iv in available if iv[0] <= unc and iv[1] > cur and iv[1] >= unc]
            if not cand:
                return -1
            cur = cand[0][1]                          # SAI: lấy đoạn đầu tiên thỏa
            ans += 1
    return ans


print("Sai :", wrong_first_fit([[1, 3], [1, 9], [3, 9]], [[1, 9]]))
print("Đúng:", sol.minSelection([[1, 3], [1, 9], [3, 9]], [[1, 9]]))

Sai : 2
Đúng: 1


## Lỗi 4 — Cố phủ cả khoảng trống giữa các mục tiêu

Nếu gộp tất cả mục tiêu thành một đoạn lớn `[min l, max r]`, ta bắt mình phủ cả những chỗ **không cần phủ** → đáp án lớn hơn hoặc sai thành `-1`. Dùng `unc = max(l, cur)` để "nhảy" qua khoảng trống.

## Lỗi 5 — Đặt lại con trỏ `i = 0` cho mỗi mục tiêu

Vẫn đúng nhưng mất `O(n · m)` → có thể TLE.

## Lỗi 6 — Đặt lại `cur` cho mỗi khối mục tiêu

Khi gộp mục tiêu thành các khối rồi xử lý từng khối, rất dễ viết `cur = "chưa phủ"` ở đầu mỗi khối. Nhưng một đoạn đã chọn có thể **phủ luôn khối sau**, nên đặt lại `cur` sẽ chọn thừa.

In [15]:
def wrong_reset_per_block(available, targets):
    ans = 0
    for L, R in merge_intervals(targets):
        cur = L - 1                                    # SAI: quên phần đã phủ từ khối trước
        while cur < R:
            unc = max(L, cur)
            best = max((b for a, b in available if a <= unc), default=-1)
            if best < unc or best <= cur:
                return -1
            cur = best
            ans += 1
    return ans


print("Sai :", wrong_reset_per_block([[1, 9]], [[1, 3], [6, 7]]))
print("Đúng:", sol.minSelection([[1, 9]], [[1, 3], [6, 7]]))

Sai : 2
Đúng: 1


# 13. Mẹo nhận diện

1. "Chọn ít đoạn nhất để phủ ..." → **sắp theo đầu trái + tham lam vươn xa nhất** (Video Stitching, Jump Game II, Minimum Number of Taps).
2. Nhiều mục tiêu rời rạc → xử lý theo đầu trái, **nhảy qua khoảng trống** bằng `max(l, cur)`.
3. Đề nói "số thực" / "điểm bất kỳ" → cẩn thận ranh giới: chạm nhau (`c <= b`) là liền, cách nhau dù 1 đơn vị là hở.
4. Tọa độ nhỏ (`<= 10^5`) → có thể thay sắp xếp bằng **mảng max tiền tố theo tọa độ**.

Các bài liên quan: *Video Stitching* (LeetCode 1024), *Minimum Number of Taps to Open to Water a Garden* (1326), *Jump Game II* (45), *Merge Intervals* (56).

# 14. Biến thể

## Biến thể 1 — Video Stitching (LeetCode 1024)

Phủ đoạn `[0, time]` bằng ít đoạn nhất — chính là bài này với một mục tiêu duy nhất.

In [16]:
def video_stitching(clips: List[List[int]], time: int) -> int:
    if time == 0:
        return 0
    return sol.minSelection(clips, [[0, time]]) if any(a <= 0 for a, _ in clips) else -1


print(video_stitching([[0, 2], [4, 6], [8, 10], [1, 9], [1, 5], [5, 9]], 10))   # 3
print(video_stitching([[0, 1], [1, 2]], 5))                                     # -1

3
-1


(Ở đây tọa độ có thể bằng 0, nên ta khởi tạo `cur = 0` không còn mang nghĩa "chưa phủ gì". Kiểm tra riêng điều kiện phải có đoạn bắt đầu tại `0`, hoặc khởi tạo `cur = -1`.)

## Biến thể 2 — Liệt kê các đoạn được chọn

Ghi lại chỉ số đoạn đạt `nxt` ở mỗi bước (xem bài tập 6).

# 15. Template tư duy

```python
avail.sort(); cur = START; i = ans = 0
for l, r in sorted(targets):
    while cur < r:
        unc = max(l, cur)                   # điểm chưa phủ đầu tiên
        nxt = cur
        while i < len(avail) and avail[i][0] <= unc:
            nxt = max(nxt, avail[i][1]); i += 1
        if nxt < unc or nxt == cur: return -1
        cur = nxt; ans += 1
return ans
```

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

```python
available = [[2,5],[1,3],[4,8],[6,9]], target = [[2,4],[7,9]]
```

Mô phỏng tham lam một con trỏ. Kiểm tra bằng cell dưới.

In [17]:
# Kiểm tra Bài 1
trace_greedy([[2, 5], [1, 3], [4, 8], [6, 9]], [[2, 4], [7, 9]]);

mục tiêu [2,4]: unc = 2, quét thêm [[1, 3], [2, 5]], chọn đoạn tới 5 -> cur = 5, ans = 1
mục tiêu [7,9]: unc = 7, quét thêm [[4, 8], [6, 9]], chọn đoạn tới 9 -> cur = 9, ans = 2


## Bài 2 — Dự đoán output

```python
print(merge_intervals([[5, 6], [1, 3], [3, 4], [6, 6], [8, 9]]))
print(covers([[1, 3], [3, 5]], [[2, 5]]), covers([[1, 3], [4, 5]], [[2, 5]]))
```

In [18]:
print(merge_intervals([[5, 6], [1, 3], [3, 4], [6, 6], [8, 9]]))
print(covers([[1, 3], [3, 5]], [[2, 5]]), covers([[1, 3], [4, 5]], [[2, 5]]))

[[1, 4], [5, 6], [8, 9]]
True False


## Bài 3 — Hoàn thiện tham lam một con trỏ

In [19]:
def exercise_3_greedy(available: List[List[int]], targets: List[List[int]]) -> int:
    avail = sorted(available)
    cur, i, ans = 0, 0, 0
    for l, r in sorted(targets):
        while cur < r:
            # TODO 1: unc = điểm chưa phủ đầu tiên
            # TODO 2: quét các đoạn có left <= unc, lấy right lớn nhất (bắt đầu từ cur)
            # TODO 3: kiểm tra không thể phủ; cập nhật cur, ans
            pass
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_greedy(*ex1) == 3
# assert exercise_3_greedy(*ex3) == -1

## Bài 4 — Jump Game II (LeetCode 45)

`nums[i]` = bước nhảy tối đa từ vị trí `i`. Tìm số bước nhảy ít nhất để tới vị trí cuối (luôn tới được).

```python
jump([2, 3, 1, 1, 4])   -> 2
jump([2, 3, 0, 1, 4])   -> 2
```

Gợi ý: từ vị trí `i` "phủ" được đoạn `[i, i + nums[i]]`.

In [20]:
def exercise_4_jump(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_jump([2, 3, 1, 1, 4]) == 2
# assert exercise_4_jump([2, 3, 0, 1, 4]) == 2

## Bài 5 — Tổng độ dài phần mục tiêu (sau khi gộp)

Tính tổng độ dài của hợp các đoạn mục tiêu (độ dài `[a, b]` là `b - a`).

```python
total_target_length([[1,3],[2,5],[7,8],[10,13],[11,15],[12,13]])   -> 4 + 1 + 5 = 10
```

In [21]:
def exercise_5_total_target_length(targets: List[List[int]]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_total_target_length([[1, 3], [2, 5], [7, 8], [10, 13], [11, 15], [12, 13]]) == 10

## Bài 6 — Mini challenge: trả về các đoạn được chọn

Trả về danh sách **chỉ số** (trong `availableIntervals` ban đầu) của một lời giải tối ưu, hoặc `[]` nếu không thể. Kiểm tra bằng `covers` và so độ dài với đáp án.

In [22]:
def exercise_6_chosen_indices(available: List[List[int]], targets: List[List[int]]) -> List[int]:
    # TODO
    raise NotImplementedError


# idx = exercise_6_chosen_indices(*ex1)
# assert len(idx) == 3 and covers([ex1[0][k] for k in idx], ex1[1])

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `unc = max(l, cur)`.
- Mức 2: `nxt = cur`; `while i < len(avail) and avail[i][0] <= unc: nxt = max(nxt, avail[i][1]); i += 1`.
- Mức 3: `if nxt < unc or nxt == cur: return -1`; `cur = nxt; ans += 1`; cuối hàm `return ans`.

**Bài 4**

- Mức 1: mục tiêu là `[0, n - 1]`, các "đoạn" là `[i, i + nums[i]]`.
- Mức 2: duyệt `i`, giữ `cur_end` (tầm của bước hiện tại) và `farthest`.
- Mức 3: khi `i == cur_end` (và chưa tới cuối): tăng bước, `cur_end = farthest`.

**Bài 5**

- Mức 1: gộp bằng `merge_intervals`.
- Mức 2: cộng `b - a` cho mỗi khối.
- Mức 3: một dòng `sum(b - a for a, b in merge_intervals(targets))`.

**Bài 6**

- Mức 1: sắp các **chỉ số** theo đầu trái: `order = sorted(range(n), key=lambda k: available[k])`.
- Mức 2: khi quét, nhớ chỉ số `best_k` đạt `nxt`.
- Mức 3: mỗi lần chọn, thêm `best_k` vào kết quả.

# 17. Đáp án tham khảo

## Đáp án Bài 1

Sắp: `[1,3], [2,5], [4,8], [6,9]`; mục tiêu `[2,4], [7,9]`.

| Mục tiêu | unc | quét thêm | nxt | cur | ans |
|---|---:|---|---:|---:|---:|
| `[2,4]` | 2 | `[1,3]`, `[2,5]` | 5 | 5 | 1 |
| `[7,9]` | 7 | `[4,8]`, `[6,9]` | 9 | 9 | 2 |

Đáp án `2` (khoảng `(5, 7)` không cần phủ).

## Đáp án Bài 2

```text
[[1, 4], [5, 6], [8, 9]]
True False
```

In [23]:
# Đáp án Bài 3
def answer_3_greedy(available: List[List[int]], targets: List[List[int]]) -> int:
    avail = sorted(available)
    cur, i, ans = 0, 0, 0
    for l, r in sorted(targets):
        while cur < r:
            unc = max(l, cur)
            nxt = cur
            while i < len(avail) and avail[i][0] <= unc:
                nxt = max(nxt, avail[i][1])
                i += 1
            if nxt < unc or nxt == cur:
                return -1
            cur = nxt
            ans += 1
    return ans


assert answer_3_greedy(*ex1) == 3
assert answer_3_greedy(*ex3) == -1
print("Bài 3: PASS")

Bài 3: PASS


In [24]:
# Đáp án Bài 4
def answer_4_jump(nums: List[int]) -> int:
    steps, cur_end, farthest = 0, 0, 0
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == cur_end:
            steps += 1
            cur_end = farthest
    return steps


assert answer_4_jump([2, 3, 1, 1, 4]) == 2
assert answer_4_jump([2, 3, 0, 1, 4]) == 2
assert answer_4_jump([0]) == 0
print("Bài 4: PASS")

Bài 4: PASS


In [25]:
# Đáp án Bài 5
def answer_5_total_target_length(targets: List[List[int]]) -> int:
    return sum(b - a for a, b in merge_intervals(targets))


assert answer_5_total_target_length([[1, 3], [2, 5], [7, 8], [10, 13], [11, 15], [12, 13]]) == 10
print("Bài 5: PASS")

Bài 5: PASS


In [26]:
# Đáp án Bài 6
def answer_6_chosen_indices(available: List[List[int]], targets: List[List[int]]) -> List[int]:
    order = sorted(range(len(available)), key=lambda k: available[k])
    cur, i = 0, 0
    chosen = []
    for l, r in sorted(targets):
        while cur < r:
            unc = max(l, cur)
            nxt, best_k = cur, -1
            while i < len(order) and available[order[i]][0] <= unc:
                if available[order[i]][1] > nxt:
                    nxt, best_k = available[order[i]][1], order[i]
                i += 1
            if nxt < unc or nxt == cur:
                return []
            cur = nxt
            chosen.append(best_k)
    return chosen


idx = answer_6_chosen_indices(*ex1)
print(idx, [ex1[0][k] for k in idx])
assert len(idx) == 3 and covers([ex1[0][k] for k in idx], ex1[1])
random.seed(6)
for _ in range(1000):
    av = []
    for _ in range(random.randint(1, 6)):
        a = random.randint(1, 8)
        av.append([a, random.randint(a, 9)])
    tg = []
    for _ in range(random.randint(1, 4)):
        a = random.randint(1, 8)
        tg.append([a, random.randint(a, 9)])
    best = sol.minSelection(av, tg)
    ids = answer_6_chosen_indices(av, tg)
    if best == -1:
        assert ids == []
    else:
        assert len(ids) == best and covers([av[k] for k in ids], tg)
print("Bài 6: PASS (1000 test: đúng số đoạn và phủ được)")

[1, 3, 4] [[1, 7], [5, 10], [10, 15]]
Bài 6: PASS (1000 test: đúng số đoạn và phủ được)


# 18. Tổng kết

**Minimum Interval Selection to Cover Target Intervals**:

1. Xử lý mục tiêu theo đầu trái; điểm chưa phủ đầu tiên là `unc = max(l, cur)` (nhảy qua khoảng trống).
2. Luôn chọn đoạn **chứa `unc`** và **vươn xa nhất** — đúng nhờ lập luận hoán đổi.
3. Không đoạn nào chứa `unc`, hoặc không tiến được → `-1`.
4. Một con trỏ trên danh sách đoạn đã sắp → `O(n log n + m log m)`.

## Độ phức tạp

```text
Time:  O(n log n + m log m)
Space: O(1) thêm (ngoài sắp xếp)
```

# 19. Checklist trước khi submit

- [ ] Đoạn chạm nhau (`c <= b`) là liền; cách nhau là **hở** (phủ số thực)?
- [ ] `unc = max(l, cur)` để không phủ khoảng trống giữa các mục tiêu?
- [ ] Kiểm tra cả `nxt < unc` **và** `nxt == cur`?
- [ ] Con trỏ `i` chỉ tiến, không đặt lại?
- [ ] Mục tiêu là một điểm `[x, x]` được xử lý đúng?
- [ ] Đã test: chạm nhau, hở, một đoạn phủ nhiều mục tiêu, không thể phủ?